# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

In [2]:
# Your code goes here

! pip install pandas

import pandas as pd

# Load all 3 files
url1 = 'https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv'
url2 = 'https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv'
url3 = 'https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv'

Mock1 = pd.read_csv(url1)
Mock2 = pd.read_csv(url2)
Mock3 = pd.read_csv(url3)

print("Shapes:", Mock1.shape, Mock2.shape, Mock3.shape)
print("Total rows before clean:", len(Mock1) + len(Mock2) + len(Mock3))

  Using cached pandas-3.0.2-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.4.4-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
  Using cached tzdata-2026.1-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-3.0.2-cp313-cp313-win_amd64.whl (9.7 MB)
Using cached numpy-2.4.4-cp313-cp313-win_amd64.whl (12.3 MB)
Using cached tzdata-2026.1-py2.py3-none-any.whl (348 kB)

   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1

In [3]:
def clean_insurance_df(df):
    """Full cleaning: cols, inconsistencies, nulls, dups, types"""
    
    # Col names: lowercase, _, st→state
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('st', 'state')
    
    # Inconsistencies (your maps)
    cleaning_maps = {
        'gender': {'femal': 'F', 'male': 'M', 'female': 'F'},
        'state': {'az': 'Arizona', 'cali': 'California', 'wa': 'Washington'},
        'education': {'bachelors': 'Bachelor'},
        'vehicle_class': {'sports car': 'Luxury', 'luxury suv': 'Luxury', 'luxury car': 'Luxury'}
    }
    for col, mapping in cleaning_maps.items():
        if col in df.columns:
            df[col] = df[col].replace(mapping)
    
    # CLV: % → numeric
    if 'customer_lifetime_value' in df.columns:
        df['customer_lifetime_value'] = df['customer_lifetime_value'].astype(str).str.replace('%', '')
        df['customer_lifetime_value'] = pd.to_numeric(df['customer_lifetime_value'], errors='coerce')
    
    # Complaints: "1/5/00" → 5
    if 'number_of_open_complaints' in df.columns:
        df['number_of_open_complaints'] = df['number_of_open_complaints'].astype(str).str.split('/').str[1]
        df['number_of_open_complaints'] = pd.to_numeric(df['number_of_open_complaints'], errors='coerce')
    
    # Nulls: median numeric, mode cat
    numeric_cols = df.select_dtypes(include=['number']).columns
    for col in numeric_cols:
        df[col] = df[col].fillna(df[col].median())
    cat_cols = df.select_dtypes(include=['object']).columns
    for col in cat_cols:
        df[col] = df[col].fillna(df[col].mode()[0])
    
    # Dups
    df = df.drop_duplicates(keep='first').reset_index(drop=True)
    
    # Numerics → int where possible
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    
    return df

In [8]:

# 1. Load with error handling
urls = {
    'mock1': 'https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv',
    'mock2': 'https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv',
    'mock3': 'https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv'
}

mock1, mock2, mock3 = None, None, None
for name, url in urls.items():
    try:
        df = pd.read_csv(url)
        print(f"✅ {name}: {df.shape}")
        globals()[name] = df  # Assign to mock1 etc.
    except Exception as e:
        print(f"❌ {name}: {e}")

print("\nLoaded shapes:", mock1.shape if mock1 is not None else "None", 
      mock2.shape if mock2 is not None else "None",
      mock3.shape if mock3 is not None else "None")

# 2. Clean only loaded files
loaded_dfs = [df for df in [mock1, mock2, mock3] if df is not None]
cleaned_dfs = [clean_insurance_df(df) for df in loaded_dfs]

# 3. Combine
unckle = pd.concat(cleaned_dfs, ignore_index=True)
print(f"\n🎉 Final unckle shape: {unckle.shape}")
print(unckle.head(2))
unckle.to_csv('insurance_combined_clean.csv', index=False)
print("💾 Saved!")

✅ mock1: (4008, 11)
✅ mock2: (996, 11)
✅ mock3: (7070, 11)

Loaded shapes: (4008, 11) (996, 11) (7070, 11)

🎉 Final unckle shape: (9138, 12)
  custateomer       state gender education custateomer_lifetime_value  income  \
0     RB50392  Washington      F    Master                 251459.20%       0   
1     QZ44356     Arizona      F  Bachelor                 697953.59%       0   

   monthly_premium_auto  number_of_open_complaints    policy_type  \
0                  1000                        0.0  Personal Auto   
1                    94                        0.0  Personal Auto   

   vehicle_class  total_claim_amount stateate  
0  Four-Door Car            2.704934      NaN  
1  Four-Door Car         1131.464935      NaN  
💾 Saved!


C:\Users\Desmond\AppData\Local\Temp\ipykernel_6936\1449400208.py:32: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns
C:\Users\Desmond\AppData\Local\Temp\ipykernel_6936\1449400208.py:32: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_g

# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

In [9]:
# Your code goes here
url = 'https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv'
unckle = pd.read_csv(url)

print("Shape:", unckle.shape)
print("\nColumns:", unckle.columns.tolist())
print("\nHead:\n", unckle.head())
print("\nDtypes:\n", unckle.dtypes)
print("\nNulls:\n", unckle.isnull().sum())
print("\nDuplicates:", unckle.duplicated().sum())

Shape: (10910, 27)

Columns: ['unnamed:_0', 'customer', 'state', 'customer_lifetime_value', 'response', 'coverage', 'education', 'effective_to_date', 'employmentstatus', 'gender', 'income', 'location_code', 'marital_status', 'monthly_premium_auto', 'months_since_last_claim', 'months_since_policy_inception', 'number_of_open_complaints', 'number_of_policies', 'policy_type', 'policy', 'renew_offer_type', 'sales_channel', 'total_claim_amount', 'vehicle_class', 'vehicle_size', 'vehicle_type', 'month']

Head:
    unnamed:_0 customer       state  customer_lifetime_value response  \
0           0  DK49336     Arizona              4809.216960       No   
1           1  KX64629  California              2228.525238       No   
2           2  LZ68649  Washington             14947.917300       No   
3           3  XL78013      Oregon             22332.439460      Yes   
4           4  QA50777      Oregon              9025.067525       No   

   coverage education effective_to_date employmentstatus 

1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

In [11]:
# Pivot: Sales Channel vs Total Revenue (rounded)
pivot_revenue = unckle.pivot_table(
    values='total_claim_amount',  # Adjust if revenue col different
    index='sales_channel',
    aggfunc='sum',
    margins=True  # Add 'All' row
).round(2)

print("Total Revenue by Sales Channel ($):\n")
print(pivot_revenue)

Total Revenue by Sales Channel ($):

               total_claim_amount
sales_channel                    
Agent                  1810226.82
Branch                 1301204.00
Call Center             926600.82
Web                     706600.04
All                    4744631.68


In [10]:
# Monthly Premium as proxy
pivot_premium = unckle.pivot_table(
    values='monthly_premium_auto',
    index='sales_channel',
    aggfunc='sum',
    margins=True
).round(2)

print("\nTotal Premium by Channel ($):\n")
print(pivot_premium)


Total Premium by Channel ($):

               monthly_premium_auto
sales_channel                      
Agent                        386335
Branch                       280953
Call Center                  197970
Web                          151511
All                         1016769


2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

In [12]:
pivot_clv = unckle.pivot_table(
    values='customer_lifetime_value',  # CLV column
    index='gender',
    columns='education',
    aggfunc='mean',
    margins=True  # 'All' row/col
).round(2)

print("Average CLV by Gender & Education ($):\n")
print(pivot_clv)

Average CLV by Gender & Education ($):

education  Bachelor  College   Doctor  High School or Below   Master      All
gender                                                                       
F           7874.27  7748.82  7328.51               8675.22  8157.05  8071.11
M           7703.60  8052.46  7415.33               8149.69  8168.83  7963.04
All         7792.27  7900.07  7372.03               8415.29  8162.52  8018.24


## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

In [ ]:
# Your code goes here